# Probability Standardizer Notebook Guide

This notebook helps you convert non-standard confidence inputs into standardized class-probability tables using `ProbStandardizer`.

It includes:
- an **interactive user-driven workflow** (you choose input format and provide file/column settings),
- strict consistency checks that stop with clear guidance when inputs are invalid,
- worked examples for each supported format.

## Supported Input Formats

`ProbStandardizer` supports these conversion modes:

1. **crisp**: one class label per row
2. **confidence**: one selected label plus a numeric confidence value in [0, 1]
3. **binary_confidence**: one selected label plus confidence flag (`label`, `is_confident`)
4. **likert**: class score columns (row-normalized to probabilities)
5. **multi_interpreter**: class vote/count columns (same conversion math as `counts`)
6. **counts**: class count/vote columns (row-normalized to probabilities)

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

from acc_assessment.standardizer import ProbStandardizer, verify_standard_structure

In [ ]:
docs_path = Path("./README-ProbabilityStandardizer.md")
if docs_path.exists():
    display(Markdown(docs_path.read_text()))
else:
    print("Could not find README-ProbabilityStandardizer.md in repository root.")

## Interactive Conversion Workflow

Fill the configuration cell below, then run the execution cell.

The notebook will **validate consistency** and stop with clear errors when required columns/settings are missing or conflicting.

In [ ]:
def _parse_class_names(value):
    if isinstance(value, list):
        classes = [str(v).strip() for v in value if str(v).strip()]
    else:
        classes = [c.strip() for c in str(value).split(",") if c.strip()]
    if not classes:
        raise ValueError("class_names must contain at least one class")
    return classes


def _normalize_mode(mode_value):
    allowed = [
        "crisp",
        "confidence",
        "binary_confidence",
        "likert",
        "multi_interpreter",
        "counts",
    ]
    mode = str(mode_value).strip().lower()
    if mode not in allowed:
        raise ValueError(
            f"Unsupported mode: {mode!r}. Use one of: {allowed}"
        )
    return mode


def _check_required_columns(df, required, context):
    missing = [col for col in required if col not in df.columns]
    if missing:
        missing_list = ", ".join(missing)
        raise ValueError(f"{context} is missing required columns: {missing_list}")


def run_standardization(
    mode,
    df,
    class_names,
    *,
    id_col="id",
    require_unique_id=False,
    label_col="label",
    is_confident_col="is_confident",
    high_p=0.99,
    low_p=0.40,
    crisp_label_col="label",
    confidence_col="confidence",
):
    mode = _normalize_mode(mode)
    class_names = _parse_class_names(class_names)

    if mode in {"likert", "counts", "multi_interpreter"}:
        _check_required_columns(df, class_names, context=f"Input table for mode={mode}")

    if mode == "binary_confidence":
        _check_required_columns(df, [label_col, is_confident_col], context="Binary confidence input")
        unknown_labels = sorted(set(df[label_col]) - set(class_names))
        if unknown_labels:
            raise ValueError(
                "Binary confidence table contains labels not in class_names: "
                + ", ".join(map(str, unknown_labels))
            )

    if mode == "confidence":
        _check_required_columns(df, [label_col, confidence_col], context="Confidence input")
        unknown_labels = sorted(set(df[label_col]) - set(class_names))
        if unknown_labels:
            raise ValueError(
                "Confidence table contains labels not in class_names: "
                + ", ".join(map(str, unknown_labels))
            )

    if mode == "crisp":
        _check_required_columns(df, [crisp_label_col], context="Crisp input")
        unknown_labels = sorted(set(df[crisp_label_col]) - set(class_names))
        if unknown_labels:
            raise ValueError(
                "Crisp table contains labels not in class_names: "
                + ", ".join(map(str, unknown_labels))
            )

    standardizer = ProbStandardizer(
        class_names=class_names,
        id_col=id_col,
        require_unique_id=require_unique_id,
    )

    if mode == "crisp":
        output = standardizer.from_crisp(df, label_col=crisp_label_col)
    elif mode == "confidence":
        output = standardizer.from_confidence(
            df,
            label_col=label_col,
            confidence_col=confidence_col,
            id_col=id_col,
        )
    elif mode == "binary_confidence":
        output = standardizer.from_binary_confidence(
            df,
            label_col=label_col,
            is_confident_col=is_confident_col,
            high_p=float(high_p),
            low_p=float(low_p),
            id_col=id_col,
        )
    elif mode == "likert":
        output = standardizer.from_likert(df, id_col=id_col)
    elif mode == "multi_interpreter":
        output = standardizer.from_multi_interpreter(df, id_col=id_col)
    else:
        output = standardizer.from_counts(df, id_col=id_col)

    is_standard = verify_standard_structure(output, class_names)
    if not is_standard:
        raise ValueError(
            "Converted output failed standard probabilistic verification. "
            "Check class_names and source data consistency."
        )

    return output, class_names

In [ ]:
# USER CONFIGURATION
# Set MODE to one of: crisp, confidence, binary_confidence, likert, multi_interpreter, counts
MODE = None

# Comma-separated string or list, e.g. "Forest,Water,Agriculture"
CLASS_NAMES = None

# Path to your input CSV
INPUT_CSV_PATH = None

# Optional output path (if None, notebook only displays output)
OUTPUT_CSV_PATH = None

# Metadata/validation settings
ID_COL = "id"
REQUIRE_UNIQUE_ID = False

# Used for crisp mode
CRISP_LABEL_COL = "label"

# Used only for confidence mode
CONFIDENCE_COL = "confidence"

# Used only for binary_confidence mode
LABEL_COL = "label"
IS_CONFIDENT_COL = "is_confident"
HIGH_P = 0.99
LOW_P = 0.40

In [ ]:
# Interactive fallback prompts if config values are left as None
mode_value = MODE if MODE is not None else input(
    "Mode (crisp/confidence/binary_confidence/likert/multi_interpreter/counts): "
).strip()
class_value = CLASS_NAMES if CLASS_NAMES is not None else input(
    "Class names (comma-separated): "
).strip()
csv_path_value = INPUT_CSV_PATH if INPUT_CSV_PATH is not None else input(
    "Input CSV path: "
).strip()

if not csv_path_value:
    raise ValueError("No input CSV path provided. Please set INPUT_CSV_PATH or provide a prompt value.")

input_df = pd.read_csv(csv_path_value)
print("Loaded input shape:", input_df.shape)
display(input_df.head())

In [ ]:
try:
    standardized_df, class_list = run_standardization(
        mode=mode_value,
        df=input_df,
        class_names=class_value,
        id_col=ID_COL,
        require_unique_id=REQUIRE_UNIQUE_ID,
        crisp_label_col=CRISP_LABEL_COL,
        confidence_col=CONFIDENCE_COL,
        label_col=LABEL_COL,
        is_confident_col=IS_CONFIDENT_COL,
        high_p=HIGH_P,
        low_p=LOW_P,
    )

    print("Standardization succeeded.")
    print("Classes used:", class_list)
    print("Output shape:", standardized_df.shape)
    display(standardized_df.head())

    if OUTPUT_CSV_PATH:
        standardized_df.to_csv(OUTPUT_CSV_PATH, index=False)
        print(f"Saved standardized output to: {OUTPUT_CSV_PATH}")

except Exception as exc:
    print("Input consistency check failed. Please correct your settings/data and rerun.")
    print("Error details:", exc)

## Live Progress Summary

Run the next cell at any time to generate a summary of what this notebook has completed so far.

Tip: run this after Cells 7–9 (configuration, input load, and execution) to get a complete workflow summary.

In [ ]:
# On-the-fly summary of work completed so far
from IPython.display import display, Markdown

def summarize_notebook_progress():
    lines = ["## Notebook Progress Summary", ""]

    mode_text = globals().get("mode_value", None)
    if mode_text is None:
        lines.append("- Mode: not set yet")
    else:
        lines.append(f"- Mode: `{mode_text}`")

    class_text = globals().get("class_list", None)
    if class_text is None:
        class_text = globals().get("class_value", None)
    if class_text is None:
        lines.append("- Classes: not set yet")
    else:
        lines.append(f"- Classes: `{class_text}`")

    input_df_local = globals().get("input_df", None)
    if input_df_local is None:
        lines.append("- Input table: not loaded yet")
    else:
        lines.append(f"- Input table loaded: `{input_df_local.shape[0]}` rows × `{input_df_local.shape[1]}` columns")

    standardized_df_local = globals().get("standardized_df", None)
    if standardized_df_local is None:
        lines.append("- Standardization: not completed yet")
    else:
        lines.append(
            f"- Standardization completed: `{standardized_df_local.shape[0]}` rows × `{standardized_df_local.shape[1]}` columns"
        )

        classes_for_check = globals().get("class_list", None)
        if classes_for_check is not None:
            try:
                ok = verify_standard_structure(standardized_df_local, classes_for_check)
                lines.append(f"- Standard structure check: `{ok}`")
            except Exception as exc:
                lines.append(f"- Standard structure check: failed to evaluate (`{exc}`)")

    output_path = globals().get("OUTPUT_CSV_PATH", None)
    if output_path:
        lines.append(f"- Output save path configured: `{output_path}`")
    else:
        lines.append("- Output save path configured: none (display-only mode)")

    display(Markdown("\n".join(lines)))

summarize_notebook_progress()

## Format Example: Crisp

Each row has exactly one class label. The standardizer converts it to binary confidence values of 1 and 0 across class columns.

In [ ]:
crisp_df = pd.DataFrame({
    "id": [1, 2, 3],
    "label": ["Forest", "Water", "Agriculture"],
})

crisp_standardizer = ProbStandardizer(
    class_names=["Forest", "Water", "Agriculture"],
    id_col="id",
)
crisp_out = crisp_standardizer.from_crisp(crisp_df, label_col="label")
display(crisp_out)
print("Crisp output is standard:", crisp_standardizer.verify_standard_style(crisp_out))

## Format Example: Confidence

Each row has a selected class label and a numeric confidence in [0, 1]. The selected class gets that confidence and remaining probability mass is spread across other classes.

In [ ]:
confidence_df = pd.DataFrame({
    "id": [34, 35],
    "label": ["Water", "Forest"],
    "confidence": [0.6, 0.9],
})

confidence_standardizer = ProbStandardizer(["Forest", "Water", "Agriculture"])
confidence_out = confidence_standardizer.from_confidence(
    confidence_df,
    label_col="label",
    confidence_col="confidence",
)

display(confidence_out)
print("Confidence output is standard:", confidence_standardizer.verify_standard_style(confidence_out))

## Format Example: Binary Confidence

Each row has a selected label and a confidence flag. The selected label gets `high_p` or `low_p`; remaining mass is spread across other classes.

In [ ]:
binary_df = pd.DataFrame({
    "id": [100, 101, 102],
    "label": ["Forest", "Water", "Agriculture"],
    "is_confident": [True, False, True],
})

binary_standardizer = ProbStandardizer(["Forest", "Water", "Agriculture"])
binary_out = binary_standardizer.from_binary_confidence(
    binary_df,
    label_col="label",
    is_confident_col="is_confident",
    high_p=0.95,
    low_p=0.55,
)

display(binary_out)
print("Binary-confidence output is standard:", binary_standardizer.verify_standard_style(binary_out))

## Format Example: Likert

Rows contain class-wise rating scores. The standardizer normalizes each row to sum to 1.

In [ ]:
likert_df = pd.DataFrame({
    "id": [10, 11, 12],
    "Forest": [5, 2, 1],
    "Water": [1, 5, 2],
    "Agriculture": [0, 1, 7],
})

likert_standardizer = ProbStandardizer(
    class_names=["Forest", "Water", "Agriculture"],
    id_col="id",
)
likert_out = likert_standardizer.from_likert(likert_df)
display(likert_out)
print("Likert output is standard:", likert_standardizer.verify_standard_style(likert_out))

## Format Example: Multi-Interpreter

Rows contain class vote totals from multiple interpreters. `from_multi_interpreter` applies count-normalization logic.

In [ ]:
multi_df = pd.DataFrame({
    "id": [20, 21],
    "Forest": [8, 1],
    "Water": [1, 3],
    "Agriculture": [1, 6],
})

multi_standardizer = ProbStandardizer(["Forest", "Water", "Agriculture"])
multi_out = multi_standardizer.from_multi_interpreter(multi_df)
display(multi_out)
print("Multi-interpreter output is standard:", multi_standardizer.verify_standard_style(multi_out))

## Format Example: Counts

Rows contain class count totals. The standardizer normalizes each row to sum to 1.

In [ ]:
counts_df = pd.DataFrame({
    "id": [30, 31],
    "Forest": [8, 1],
    "Water": [1, 3],
    "Agriculture": [1, 6],
})

counts_standardizer = ProbStandardizer(["Forest", "Water", "Agriculture"])
counts_out = counts_standardizer.from_counts(counts_df)
display(counts_out)
print("Counts output is standard:", counts_standardizer.verify_standard_style(counts_out))

## Suggested Next Step

After producing standardized outputs here, use them in your integrated assessment workflow notebook (`AccuracyAssessmentTools-hfff.ipynb`).